# h2ml Quickstart

This notebook demonstrates the h2ml pipeline on small built-in datasets:

1. **Classification** — breast cancer dataset (binary, 30 features)
2. **Regression** — diabetes dataset (10 features, y-transform sweep)
3. **Spatio-temporal** — synthetic geodata showing spatial CV and space-time local conformal intervals

No external data required — everything runs with `sklearn.datasets` or a small synthetic generator.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_diabetes

H2ML_ENABLE_TABPFN=1
from h2ml.features.feature_store import PipelineData
from h2ml.pipeline.pipeline import H2MLPipeline, PipelineConfig
from h2ml.plots import (
    cv_diagnostics,
    pipeline_scores,
    shap_dependence,
    shap_importance,
    shap_summary_plot,
)

c:\Users\h2ugo\Documents\h2ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. Classification — breast cancer dataset

In [2]:
data = load_breast_cancer()

store_clf = PipelineData(
    X=data.data.astype(np.float32),
    feature_names=list(data.feature_names),
    y=data.target.astype(np.float32),
)

print(store_clf)

PipelineData(n_samples=569, n_features=30, features=[np.str_('mean radius'), np.str_('mean texture'), np.str_('mean perimeter')]..., has_coords=False, has_times=False)


In [3]:
config_clf = PipelineConfig(
    task_type="classification",   # or "regression"; the TaskType enum is also accepted
    metric="AUC",
    n_splits=5,
    n_trials=30,
    verbose=True,
    n_jobs=4,
)

pipeline_clf = H2MLPipeline(config=config_clf)
result_clf = pipeline_clf.run(store_clf)

2026-08-07 10:13:21.703 | INFO     | h2ml.pipeline.pipeline:_log:1207 - Step 1 — 5-fold CV 11 models × 1 transform(s) on all features → select best
2026-08-07 10:13:28.693 | INFO     | h2ml.pipeline.pipeline:_log:1207 -   Best: LogisticRegression (AUC=0.9953 ± 0.0060)
2026-08-07 10:13:28.693 | INFO     | h2ml.pipeline.pipeline:_log:1207 - Step 2 — Feature reduction (threshold=0.7)
2026-08-07 10:13:28.694 | INFO     | h2ml.features.shap_importance:get_oof_shap_values:423 - Computing OOF SHAP values for LogisticRegression (5-fold, n_samples=569)


KeyboardInterrupt: 

In [ ]:
# Results across all pipeline stages
result_clf.summary(metric="AUC_Test_Mean")

In [ ]:
# Features selected after step 2
print(f"Original features : {store_clf.n_features}")
print(f"Reduced features  : {result_clf.features_reduced.n_features}")
print(f"Kept              : {result_clf.features_reduced.feature_names}")

In [ ]:
# SHAP feature importance from step 2
result_clf.selector.importance_summary()

In [ ]:
# Build and inspect the deployment artifact
final_clf = result_clf.build_final_model()
print(final_clf)

# Predict on the training set as a sanity check
preds = final_clf.predict(store_clf.X[:, [store_clf.feature_names.index(f) for f in final_clf.feature_names]])
print(f"Prediction sample : {preds[:10]}")

### Conformal prediction sets (classification)

`build_final_model()` automatically calibrates a conformal predictor from the out-of-fold CV predictions. `predict_set(X, alpha)` returns, for each sample, the set of classes that are plausible at the requested coverage level.

- **`[1]`** or **`[0]`** — model is confident; only one class is compatible with the coverage guarantee  
- **`[0, 1]`** — model is uncertain; both classes are plausible  

Coverage is guaranteed to be ≥ `1 - alpha` on average over new draws from the training distribution.

In [ ]:
# Align X to the features the final model was trained on
feat_idx = [store_clf.feature_names.index(f) for f in final_clf.feature_names]
X_aligned = store_clf.X[:, feat_idx]

# 90% conformal prediction sets
sets = final_clf.predict_set(X_aligned, alpha=0.10)

n_confident = sum(len(s) == 1 for s in sets)
n_uncertain = sum(len(s) == 2 for s in sets)
n_empty = sum(len(s) == 0 for s in sets)

print(f"Confident   (singleton set): {n_confident} / {len(sets)}")
print(f"Uncertain   (both classes) : {n_uncertain} / {len(sets)}")
print(f"Empty set                  : {n_empty} / {len(sets)}")
print(f"\nFirst 10 prediction sets: {[s.tolist() for s in sets[:10]]}")
print(f"Calibration threshold q  : {final_clf.conformal.threshold(0.10):.4f}")

### Diagnostic plots

In [ ]:
pipeline_scores(result_clf)
cv_diagnostics(result_clf.best_cv_result)
shap_importance(result_clf.selector)
shap_summary_plot(result_clf)
shap_dependence(result_clf, n_features=6)

---
## 2. Regression — diabetes dataset with y-transform sweep

In [ ]:
data_r = load_diabetes()

store_reg = PipelineData(
    X=data_r.data.astype(np.float32),
    feature_names=list(data_r.feature_names),
    y=data_r.target.astype(np.float32),
)

print(store_reg)

In [ ]:
config_reg = PipelineConfig(
    task_type="regression",
    metric="R2",
    n_splits=5,
    n_trials=30,
    verbose=True,
    n_jobs=4,
)

pipeline_reg = H2MLPipeline(config=config_reg)

# Sweep log, sqrt, and identity transforms alongside raw y
result_reg = pipeline_reg.run(store_reg, transforms=["count", "log", "sqrt"])

### Conformal prediction intervals (regression)

`predict_interval(X, alpha)` returns symmetric `(lower, upper)` bounds around the point estimate. The interval width is constant (`2q`) — the same threshold `q` applies to every prediction.

**Note:** if a y-transform was used, the interval is in the transformed space. Apply the inverse transform to the bounds to recover original-scale intervals.

In [ ]:
final_reg = result_reg.build_final_model()

feat_idx_r = [store_reg.feature_names.index(f) for f in final_reg.feature_names]
X_aligned_r = store_reg.X[:, feat_idx_r]

# 90% prediction intervals
lower, upper = final_reg.predict_interval(X_aligned_r, alpha=0.10)
y_hat = final_reg.predict(X_aligned_r)

print(f"Calibration threshold q : {final_reg.conformal.threshold(0.10):.4f}")
print(f"Interval half-width     : ±{(upper - lower).mean() / 2:.4f}")
print()

# If a y-transform was applied, optionally invert the bounds
if result_reg.y_transform:
    from h2ml.preprocessing.transforms import INVERSE_TRANSFORMS

    inv = INVERSE_TRANSFORMS.get(result_reg.y_transform)
    if inv is not None:
        lower_orig = inv(lower)
        upper_orig = inv(upper)
        y_hat_orig = inv(y_hat)
        print(f"Transform in use        : {result_reg.y_transform}")
        print(f"Original-scale interval : [{lower_orig[:3].round(1)} … {upper_orig[:3].round(1)}]")
    else:
        print(f"No inverse transform registered for '{result_reg.y_transform}'")
else:
    print("Sample intervals (first 5 rows):")
    for i in range(5):
        print(f"  ŷ={y_hat[i]:.2f}  [{lower[i]:.2f}, {upper[i]:.2f}]")

In [ ]:
result_reg.summary(metric="R2_Test_Mean")

In [ ]:
print(f"Best model     : {result_reg.best_model_name}")
print(f"Best stage     : {result_reg.best_stage}")
print(f"Best transform : {result_reg.y_transform}")
print(f"Best params    : {result_reg.best_params}")

### Diagnostic plots

In [ ]:
pipeline_scores(result_reg)
cv_diagnostics(result_reg.best_cv_result)
shap_importance(result_reg.selector)
shap_summary_plot(result_reg)
shap_dependence(result_reg, n_features=6)

---
## 3. Spatio-temporal — spatial CV & local conformal

When `store.coords` (and optionally `store.times`) are provided, the pipeline automatically switches to **spatial cross-validation** — holding out whole geographic blocks instead of random rows, so the score isn't inflated by spatial autocorrelation. With `store.times` too, conformal calibration becomes **block × season local**: the prediction interval adapts to *where* and *when* each point falls.

Below we build a synthetic dataset over the Iberian peninsula. The model's features (`temperature`, `precip`, `elevation`) carry the learnable signal, while a latitude gradient and a seasonal term are *left out of the features on purpose* — so they surface as spatially and seasonally structured residuals that local conformal calibration picks up. Still no external data required.

In [ ]:
# Synthetic spatio-temporal regression over the Iberian peninsula bounding box.
rng = np.random.default_rng(0)
n = 800

lat = rng.uniform(36.0, 44.0, n)
lon = rng.uniform(-9.0, 3.0, n)
coords = np.column_stack([lat, lon])                       # (n, 2) lat/lon

start = np.datetime64("2021-01-01")
times = (start + rng.integers(0, 730, n).astype("timedelta64[D]")).astype("datetime64[D]")

# Seasonal phase: +1 mid-summer, -1 mid-winter
doy = (times - times.astype("datetime64[Y]")).astype(int)
season = np.sin(2 * np.pi * (doy - 80) / 365.25)

# Informative covariates -> these become the model's features
elevation   = rng.uniform(0, 2000, n)
temperature = 22 - 0.6 * (lat - 36) - 0.004 * elevation + 6 * season + rng.normal(0, 1.0, n)
precip      = rng.gamma(2.0, 1.0, n) + 0.3 * (lat - 36)
noise_a     = rng.standard_normal(n)                       # uninformative -> dropped in step 2
noise_b     = rng.standard_normal(n)

X_st = np.column_stack([temperature, precip, elevation, noise_a, noise_b]).astype(np.float32)
feature_names_st = ["temperature", "precip", "elevation", "noise_a", "noise_b"]

# Target = learnable signal from features + a latitude gradient and a seasonal term that are
# deliberately NOT features, so they surface as spatially/seasonally structured residuals.
# Heteroscedastic noise (larger in the north and in summer) is what local conformal captures.
noise_scale = 1.0 + 0.5 * (lat - 36) / 8 + 1.0 * np.clip(season, 0, None)
y_st = (
    1.8 * temperature - 1.2 * precip + 0.002 * elevation
    + 1.5 * (lat - 40.0)            # spatial gradient (not a feature)
    + 2.0 * season                 # seasonal term  (not a feature)
    + rng.normal(0, noise_scale)
).astype(np.float32)

store_st = PipelineData(
    X=X_st,
    feature_names=feature_names_st,
    y=y_st,
    coords=coords,   # (n, 2) -> activates spatial cross-validation
    times=times,     # (n,)   -> activates space-time local conformal calibration
)
print(store_st)

In [ ]:
config_st = PipelineConfig(
    task_type="regression",
    metric="R2",
    n_splits=5,
    n_trials=20,
    spatial_cv_method="block",    # deterministic block grid (vs "spcv" AHC clusters)
    n_blocks_per_fold=2,          # ~16 spatial blocks, large enough for per-season cells
    time_bin_resolution="season", # compound conformal cells binned DJF/MAM/JJA/SON
    verbose=False,
)

result_st = H2MLPipeline(config=config_st).run(store_st)

print("cv_type        :", result_st.cv_type)           # -> "spatial"
print("splitter       :", type(result_st.splitter).__name__)
print("spatial blocks :", len(np.unique(result_st.splitter.block_id_)))
print("best model     :", result_st.best_model_name)
print("best R2 (test) :", round(result_st.summary(metric="R2_Test_Mean")["R2_Test_Mean"].max(), 3))

### Spatial CV blocks

`splitter.plot()` visualises the geographic blocks the model is scored on out-of-sample: the left panel colours each point by its block, the right panel by the fold that holds that block out.

In [ ]:
# Left: each point coloured by its spatial block. Right: which fold holds out that block.
result_st.splitter.plot()

### Local conformal summary

`build_final_model()` calibrates a **space-time local** conformal predictor from the out-of-fold residuals. `local_conformal.summary()` reports, per spatial block and per season, the calibration sample count `n` and the nonconformity quantile `q` (the interval half-width that block/season would apply). The `used` column shows which fallback level each cell resolves to — `compound` (block × season), `block` (season pooled), or `global`.

In [ ]:
final_st = result_st.build_final_model()

# One 'spatial' row per block + one 'compound' row per (block, season) cell.
# bin_name follows time_bin_resolution; 'used' shows the fallback level each cell resolves to.
summary_st = final_st.local_conformal.summary(alpha=0.10)
summary_st.head(12)

### Spatio-temporally adaptive intervals

Passing `coords` and `times` to `predict_interval` makes each interval's width vary by **location** and **season** — wider where and when residuals are larger. Contrast this with the constant-width intervals in the non-spatial regression section above, where a single global `q` applies to every prediction.

In [ ]:
# Passing coords + times makes the interval width adapt by location and season
feat_idx_st = [store_st.feature_names.index(f) for f in final_st.feature_names]
X_st = store_st.X[:, feat_idx_st]

lower_st, upper_st = final_st.predict_interval(
    X_st, alpha=0.10, coords=store_st.coords, times=store_st.times
)
half_width = (upper_st - lower_st) / 2

print(f"interval half-width  min={half_width.min():.2f}  "
      f"mean={half_width.mean():.2f}  max={half_width.max():.2f}")
print("first 8 half-widths:", half_width[:8].round(2))

---
## 4. Partial run — step 1 only (quick model screening)

In [ ]:
# Useful when you just want to compare models before committing to the full pipeline
result_screen = pipeline_clf.run_step1_only(store_clf)
result_screen.step1_agg_df.sort_values("AUC_Test_Mean", ascending=False)

---
## 5. Persistence

In [ ]:
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)

# Save the full pipeline result
result_clf.save(out / "clf_result")

# Save the deployment model
final_clf.save(out / "clf_final_model.pkl")

# Reload
from h2ml.pipeline.pipeline import PipelineResult  # noqa: E402
from h2ml.pipeline.final_model import FinalModel  # noqa: E402

result_reloaded = PipelineResult.load(out / "clf_result")
model_reloaded = FinalModel.load(out / "clf_final_model.pkl")

print(result_reloaded)
print(model_reloaded)